# 02 Naive Baseline Model

This notebook implements the naive baseline for **RQ2**.

## Objective
Predict song popularity using a simple mean-based baseline and evaluate it with:
- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R²

## Research Question
**RQ2:** How accurately can song popularity be predicted using danceability, energy, speechiness, tempo, valence, and duration_ms?


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## 1. Define project paths

In [ ]:
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"

TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data dir     :", DATA_DIR)
print("Tables dir   :", TABLES_DIR)


## 2. Load and clean the track-level dataset

In [ ]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Standardize column names and remove unnamed index-like columns.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with original column names.

    Returns
    -------
    pd.DataFrame
        Copy of the dataframe with cleaned column names.
    """
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    unnamed_cols = [c for c in df.columns if c.startswith("unnamed")]
    if unnamed_cols:
        df = df.drop(columns=unnamed_cols)
    return df

tracks = pd.read_csv(DATA_DIR / "spotify_tracks.csv")
tracks = clean_columns(tracks)

print("Tracks shape:", tracks.shape)
display(tracks.head(3))


## 3. Select variables for RQ2

In [ ]:
rq2_vars = [
    "popularity",
    "danceability",
    "energy",
    "speechiness",
    "tempo",
    "valence",
    "duration_ms"
]

df = tracks[rq2_vars].copy().dropna()

print("=== RQ2 DATASET INFO ===")
print("Shape:", df.shape)
display(df.head(3))


## 4. Train-test split

The target variable is `popularity`, which is continuous. Therefore, the task is formulated as a regression problem.


In [ ]:
X = df.drop(columns=["popularity"])
y = df["popularity"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))


## 5. Define the naive baseline

The baseline predicts the **mean popularity in the training set** for every observation in the test set.


In [ ]:
baseline_value = y_train.mean()
y_pred_baseline = np.full(shape=len(y_test), fill_value=baseline_value)

print("Baseline mean popularity:", round(baseline_value, 3))


## 6. Evaluate baseline performance

In [ ]:
mae = mean_absolute_error(y_test, y_pred_baseline)
rmse = mean_squared_error(y_test, y_pred_baseline, squared=False)
r2 = r2_score(y_test, y_pred_baseline)

print("=== NAIVE BASELINE RESULTS ===")
print("MAE :", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("R^2 :", round(r2, 3))


## 7. Create result tables

The first table stores the three evaluation metrics.  
The second table stores summary information about the baseline experiment.


In [ ]:
results_df = pd.DataFrame({
    "metric": ["MAE", "RMSE", "R2"],
    "naive_baseline_score": [mae, rmse, r2]
})

summary_df = pd.DataFrame({
    "item": ["dataset_rows", "train_size", "test_size", "baseline_mean_popularity"],
    "value": [len(df), len(X_train), len(X_test), baseline_value]
})

display(results_df)
display(summary_df)


## 8. Save outputs

In [ ]:
results_df.to_csv(TABLES_DIR / "step4_naive_baseline_results.csv", index=False)
summary_df.to_csv(TABLES_DIR / "step4_baseline_summary.csv", index=False)

print("Files saved:")
print(TABLES_DIR / "step4_naive_baseline_results.csv")
print(TABLES_DIR / "step4_baseline_summary.csv")


## 9. Interpretation

A mean-based baseline is expected to have an R² value close to zero because it does not use any explanatory variables.  
The main purpose of this notebook is to establish a lower-bound benchmark. In the final phase, trained regression models should outperform this baseline by reducing MAE and RMSE and by achieving a stronger model fit.
